In [7]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import os
  
# fetch dataset 
statlog_german_credit_data = fetch_ucirepo(id=144)

# data (as pandas dataframes) 
X = statlog_german_credit_data.data.features 
y = statlog_german_credit_data.data.targets 

# metadata 
print(statlog_german_credit_data.metadata) 
  
# variable information 
print(statlog_german_credit_data.variables)

{'uci_id': 144, 'name': 'Statlog (German Credit Data)', 'repository_url': 'https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data', 'data_url': 'https://archive.ics.uci.edu/static/public/144/data.csv', 'abstract': 'This dataset classifies people described by a set of attributes as good or bad credit risks. Comes in two formats (one all numeric). Also comes with a cost matrix', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 1000, 'num_features': 20, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Other', 'Marital Status', 'Age', 'Occupation'], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1994, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5NC77', 'creators': ['Hans Hofmann'], 'intro_paper': None, 'additional_info': {'summary': 'Two datasets are provided.  the original dataset, in the form provided by

In [ ]:
column_names = {
    "Attribute1": "checking_status",
    "Attribute2": "duration_months",
    "Attribute3": "credit_history",
    "Attribute4": "purpose",
    "Attribute5": "credit_amount",
    "Attribute6": "savings",
    "Attribute7": "employment_since",
    "Attribute8": "installment_rate",
    "Attribute9": "personal_status_sex",
    "Attribute10": "other_debtors",
    "Attribute11": "residence_since",
    "Attribute12": "property",
    "Attribute13": "age_years",
    "Attribute14": "other_installment_plans",
    "Attribute15": "housing",
    "Attribute16": "existing_credits",
    "Attribute17": "job",
    "Attribute18": "dependents",
    "Attribute19": "telephone",
    "Attribute20": "foreign_worker",
}

X = X.rename(columns=column_names)
y = y.rename(columns={"class": "credit_risk"})

print(X.columns.tolist())
print(y.columns.tolist())

['checking_status', 'duration_months', 'credit_history', 'purpose', 'credit_amount', 'savings', 'employment_since', 'installment_rate', 'personal_status_sex', 'other_debtors', 'residence_since', 'property', 'age_years', 'other_installment_plans', 'housing', 'existing_credits', 'job', 'dependents', 'telephone', 'foreign_worker']
['credit_risk']


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Create a pipeline with preprocessing and the classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Train the model
pipeline.fit(X_train, y_train.values.ravel())
y_pred = pipeline.predict(X_test)

# Evaluate the model
accuracy = pipeline.score(X_test, y_test)
print(f"Model accuracy: {accuracy:.2f}")

Model accuracy: 0.80


In [17]:
import pandas as pd
from sklearn.metrics import confusion_matrix

# Compute fairness measures: male vs female
male_codes = ['A91', 'A93', 'A94']
female_codes = ['A92', 'A95']

# Create gender binary variable from personal_status_sex
gender = data['personal_status_sex'].apply(lambda x: 'Male' if x in male_codes else 'Female')

# Get model predictions and true labels
y_true = y_test.values
y_pred_test = y_pred  # From earlier model predictions

# Separate predictions by gender
male_mask = gender.iloc[X_test.index] == 'Male'
female_mask = gender.iloc[X_test.index] == 'Female'

y_true_male = y_true[male_mask]
y_pred_male = y_pred_test[male_mask]

y_true_female = y_true[female_mask]
y_pred_female = y_pred_test[female_mask]

# Calculate True Positive Rate (TPR) for each group
def calculate_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    return tpr

tpr_male = calculate_tpr(y_true_male, y_pred_male)
tpr_female = calculate_tpr(y_true_female, y_pred_female)

# Compute Equal Opportunity metrics
equal_opportunity_diff = abs(tpr_male - tpr_female)
equal_opportunity_ratio = min(tpr_male, tpr_female) / max(tpr_male, tpr_female) if max(tpr_male, tpr_female) > 0 else 0

print(f"Male TPR (True Positive Rate):     {tpr_male:.4f}")
print(f"Female TPR (True Positive Rate):   {tpr_female:.4f}")

KeyError: 'personal_status_sex'